<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Series temporales, rolling, resampling y feature engineering con Pandas y Polars**

## Pandas avanzado + Polars

Este laboratorio cierra el bloque de manipulación tabular avanzada centrándose en datos temporales.

Trabajaremos con una idea central:

> **en series temporales, el orden importa y cualquier transformación puede introducir leakage si utiliza información futura.**

La secuencia será:

> **observar → detectar anomalía o riesgo → construir la transformación → visualizar → validar temporalmente → comparar Pandas/Polars → medir**

El laboratorio incluye:

- fechas y tipos temporales;
- orden temporal;
- frecuencia;
- resampling;
- rolling;
- expanding;
- ventanas temporales;
- lags;
- diferencias;
- tasas de cambio;
- features calendáricas;
- detección de anomalías;
- leakage;
- gaps temporales;
- Polars eager;
- Polars lazy;
- ventanas dinámicas;
- benchmarks;
- validación de equivalencia.

## Objetivos

Al finalizar deberías poder:

1. Inspeccionar columnas temporales.
2. Convertir strings a datetime.
3. Detectar fechas inválidas.
4. Ordenar correctamente series temporales.
5. Detectar huecos y duplicados temporales.
6. Reagrupar por frecuencia con `resample`.
7. Utilizar rolling windows.
8. Utilizar expanding windows.
9. Crear lags y diferencias.
10. Construir features temporales.
11. Detectar leakage temporal.
12. Diseñar ventanas que solo usen información pasada.
13. Detectar anomalías mediante rolling statistics.
14. Trabajar con ventanas temporales en Polars.
15. Utilizar `group_by_dynamic`.
16. Utilizar `rolling` y `over`.
17. Implementar pipelines lazy.
18. Comparar Pandas y Polars con benchmarks.
19. Validar equivalencia temporal entre motores.

## 1. Preparación

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

try:
    import polars as pl
    POLARS_DISPONIBLE = True
    print("Polars:", pl.__version__)
except ImportError:
    POLARS_DISPONIBLE = False
    print("Polars no instalado. Ejecuta: pip install polars")

print("Pandas:", pd.__version__)

# 2. Crear una serie temporal de trabajo

In [ ]:
n_horas = 24 * 180

fechas = pd.date_range(
    "2025-01-01",
    periods=n_horas,
    freq="h"
)

t = np.arange(n_horas)

serie = pd.DataFrame({
    "timestamp": fechas,
    "region": rng.choice(
        ["Norte", "Sur", "Este"],
        n_horas,
        p=[0.40, 0.35, 0.25]
    ),
    "demanda": (
        100
        + 15 * np.sin(2 * np.pi * t / 24)
        + 10 * np.sin(2 * np.pi * t / (24 * 7))
        + rng.normal(0, 6, n_horas)
    )
})

serie.head()

### Interpretación

La variable `demanda` contiene:

- patrón diario;
- patrón semanal;
- ruido.

Esto nos permitirá analizar estacionalidad, tendencias locales y anomalías.

# 3. Primera visualización temporal

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

muestra = serie.head(24 * 14)

ax.plot(
    muestra["timestamp"],
    muestra["demanda"]
)

ax.set(
    title="Demanda horaria — primeras dos semanas",
    xlabel="Fecha",
    ylabel="Demanda"
)

plt.show()

### Interpretación

Se aprecia un patrón periódico diario. Antes de aplicar cualquier resumen temporal debemos comprender cómo evoluciona la señal.

# 4. Introducir inconsistencias temporales deliberadas

In [ ]:
serie_sucia = serie.copy()

# Duplicado temporal
fila_dup = serie_sucia.iloc[[100]].copy()
serie_sucia = pd.concat(
    [serie_sucia, fila_dup],
    ignore_index=True
)

# Huecos
indices_eliminar = [200, 201, 202, 800, 1500]
serie_sucia = serie_sucia.drop(
    index=indices_eliminar
).reset_index(drop=True)

# Fecha inválida representada como string en copia separada
serie_strings = serie_sucia.copy()
serie_strings["timestamp"] = (
    serie_strings["timestamp"]
    .astype(str)
)

serie_strings.loc[10, "timestamp"] = "fecha_invalida"

print("Filas originales:", len(serie))
print("Filas sucias:", len(serie_sucia))

# 5. Detectar tipo incorrecto

In [ ]:
print(serie_strings.dtypes)

La columna temporal ha pasado a `object`.

Esto es una señal de que Pandas ya no puede aplicar correctamente operaciones temporales avanzadas.

# 6. Convertir y detectar fecha inválida

In [ ]:
serie_parseada = serie_strings.copy()

serie_parseada["timestamp"] = pd.to_datetime(
    serie_parseada["timestamp"],
    errors="coerce"
)

print(
    "Fechas inválidas:",
    serie_parseada["timestamp"].isna().sum()
)

### Interpretación

`errors="coerce"` convierte fechas no interpretables en `NaT`.

La inconsistencia queda visible y cuantificable.

# 7. Inspeccionar la fecha inválida

In [ ]:
serie_parseada.loc[
    serie_parseada["timestamp"].isna()
]

# 8. Corregir eliminando el registro inválido

In [ ]:
serie_parseada = (
    serie_parseada
    .dropna(subset=["timestamp"])
    .copy()
)

print(
    "Fechas inválidas después:",
    serie_parseada["timestamp"].isna().sum()
)

# 9. Detectar duplicados temporales

In [ ]:
duplicados_temporales = (
    serie_parseada
    .duplicated(
        subset=["timestamp", "region"],
        keep=False
    )
)

print(
    "Filas duplicadas temporalmente:",
    duplicados_temporales.sum()
)

In [ ]:
serie_parseada.loc[
    duplicados_temporales
].sort_values(
    ["timestamp", "region"]
).head()

### Interpretación

Un timestamp duplicado no siempre es incorrecto.

Debemos preguntarnos:

- ¿la granularidad debería ser una fila por timestamp?
- ¿una fila por timestamp y región?
- ¿una fila por sensor?

La clave temporal debe interpretarse junto con la granularidad.

# 10. Orden temporal

In [ ]:
serie_desordenada = (
    serie_parseada
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(drop=True)
)

print(
    serie_desordenada[
        ["timestamp", "demanda"]
    ].head()
)

Las funciones `rolling`, `diff`, `shift` o `merge_asof` dependen del orden temporal.

Aplicarlas sobre datos desordenados puede producir resultados conceptualmente incorrectos.

# 11. Corregir orden

In [ ]:
serie_ordenada = (
    serie_parseada
    .sort_values("timestamp")
    .reset_index(drop=True)
)

serie_ordenada.head()

# 12. Detectar gaps temporales

In [ ]:
serie_unica = (
    serie_ordenada
    .drop_duplicates(
        subset="timestamp"
    )
    .sort_values("timestamp")
)

serie_unica["delta_t"] = (
    serie_unica["timestamp"]
    .diff()
)

serie_unica[
    serie_unica["delta_t"]
    > pd.Timedelta(hours=1)
][
    ["timestamp", "delta_t"]
].head(10)

### Interpretación

Una diferencia superior a una hora indica que faltan observaciones respecto a la frecuencia esperada.

# 13. Visualizar gaps

In [ ]:
gaps = serie_unica.loc[
    serie_unica["delta_t"]
    > pd.Timedelta(hours=1)
]

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    gaps["timestamp"].astype(str),
    gaps["delta_t"].dt.total_seconds() / 3600
)

ax.set(
    title="Huecos temporales detectados",
    xlabel="Timestamp posterior al gap",
    ylabel="Duración del gap (horas)"
)

ax.tick_params(axis="x", rotation=45)

plt.show()

# 14. Construir un índice temporal completo

In [ ]:
serie_indexada = (
    serie_unica
    .set_index("timestamp")
)

indice_completo = pd.date_range(
    serie_indexada.index.min(),
    serie_indexada.index.max(),
    freq="h"
)

serie_reindexada = (
    serie_indexada
    .reindex(indice_completo)
)

serie_reindexada.index.name = "timestamp"

print(
    "Huecos materializados como NaN:",
    serie_reindexada["demanda"].isna().sum()
)

### Interpretación

Reindexar a una frecuencia conocida hace explícitos los huecos.

Esto es preferible a dejar discontinuidades invisibles.

# 15. Visualizar los valores faltantes

In [ ]:
tramo = serie_reindexada.iloc[
    180:230
]

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(
    tramo.index,
    tramo["demanda"],
    marker="o"
)

ax.set(
    title="Tramo con hueco temporal",
    xlabel="Fecha",
    ylabel="Demanda"
)

plt.show()

# 16. Imputación temporal: primero discutir

No existe una única imputación correcta.

Opciones habituales:

- forward fill;
- backward fill;
- interpolación;
- modelo temporal;
- dejar missing.

Para esta práctica utilizaremos interpolación temporal.

# 17. Interpolación temporal

In [ ]:
serie_interpolada = serie_reindexada.copy()

serie_interpolada["demanda"] = (
    serie_interpolada["demanda"]
    .interpolate(
        method="time"
    )
)

print(
    "Nulos restantes:",
    serie_interpolada["demanda"].isna().sum()
)

# 18. Antes vs después

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(
    tramo.index,
    tramo["demanda"],
    marker="o",
    label="Antes"
)

ax.plot(
    serie_interpolada.iloc[180:230].index,
    serie_interpolada.iloc[180:230]["demanda"],
    marker=".",
    label="Interpolado"
)

ax.set(
    title="Tratamiento del hueco temporal",
    xlabel="Fecha",
    ylabel="Demanda"
)

ax.legend()
plt.show()

### Interpretación

La interpolación reconstruye continuidad, pero introduce valores estimados.

En datos reales esta decisión debe quedar documentada y validada.

# 19. Resampling diario

In [ ]:
diario = (
    serie_interpolada["demanda"]
    .resample("D")
    .agg(
        ["mean", "min", "max", "sum"]
    )
)

diario.head()

## 19.1 Visualizar media diaria

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    diario.index,
    diario["mean"]
)

ax.set(
    title="Demanda media diaria",
    xlabel="Fecha",
    ylabel="Demanda media"
)

plt.show()

# 20. Resampling semanal

In [ ]:
semanal = (
    serie_interpolada["demanda"]
    .resample("W")
    .agg(
        media="mean",
        total="sum",
        maximo="max"
    )
)

semanal.head()

### Diferencia conceptual

Cambiar frecuencia implica una agregación.

No estamos simplemente “reduciendo filas”: estamos definiendo cómo representar un periodo completo.

# 21. Rolling window

In [ ]:
serie_roll = (
    serie_interpolada
    .reset_index()
)

serie_roll["media_24h"] = (
    serie_roll["demanda"]
    .rolling(
        window=24,
        min_periods=1
    )
    .mean()
)

serie_roll[[
    "timestamp",
    "demanda",
    "media_24h"
]].head(30)

## 21.1 Visualizar rolling

In [ ]:
muestra = serie_roll.head(
    24 * 14
)

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    muestra["timestamp"],
    muestra["demanda"],
    alpha=0.45,
    label="Demanda"
)

ax.plot(
    muestra["timestamp"],
    muestra["media_24h"],
    label="Media móvil 24h"
)

ax.set(
    title="Demanda y media móvil de 24 horas",
    xlabel="Fecha",
    ylabel="Demanda"
)

ax.legend()
plt.show()

### Interpretación

La media móvil suaviza la variabilidad de corto plazo y permite observar mejor la evolución local.

# 22. Rolling basado en tiempo

In [ ]:
serie_tiempo = (
    serie_roll
    .set_index("timestamp")
)

serie_tiempo["media_24h_tiempo"] = (
    serie_tiempo["demanda"]
    .rolling("24h")
    .mean()
)

serie_tiempo.head()

Las ventanas por número de filas y por duración temporal no son siempre equivalentes si existen gaps o frecuencias irregulares.

# 23. Expanding window

In [ ]:
serie_roll["media_acumulada"] = (
    serie_roll["demanda"]
    .expanding()
    .mean()
)

serie_roll[[
    "timestamp",
    "demanda",
    "media_acumulada"
]].head()

## 23.1 Visualizar expanding

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

muestra_exp = serie_roll.head(
    24 * 30
)

ax.plot(
    muestra_exp["timestamp"],
    muestra_exp["demanda"],
    alpha=0.3,
    label="Demanda"
)

ax.plot(
    muestra_exp["timestamp"],
    muestra_exp["media_acumulada"],
    label="Media acumulada"
)

ax.legend()
ax.set(
    title="Expanding mean",
    xlabel="Fecha",
    ylabel="Demanda"
)

plt.show()

# 24. Lags

In [ ]:
serie_roll["lag_1h"] = (
    serie_roll["demanda"]
    .shift(1)
)

serie_roll["lag_24h"] = (
    serie_roll["demanda"]
    .shift(24)
)

serie_roll[[
    "timestamp",
    "demanda",
    "lag_1h",
    "lag_24h"
]].head(30)

### Interpretación

Los lags permiten utilizar valores históricos como features.

Un lag bien construido utiliza únicamente información disponible en el pasado.

# 25. Diferencias

In [ ]:
serie_roll["diff_1h"] = (
    serie_roll["demanda"]
    .diff(1)
)

serie_roll["diff_24h"] = (
    serie_roll["demanda"]
    .diff(24)
)

serie_roll[[
    "timestamp",
    "demanda",
    "diff_1h",
    "diff_24h"
]].head(30)

# 26. Cambio porcentual

In [ ]:
serie_roll["pct_change_1h"] = (
    serie_roll["demanda"]
    .pct_change()
)

serie_roll[[
    "timestamp",
    "demanda",
    "pct_change_1h"
]].head()

# 27. Features calendáricas

In [ ]:
serie_roll["hora"] = (
    serie_roll["timestamp"]
    .dt.hour
)

serie_roll["dia_semana"] = (
    serie_roll["timestamp"]
    .dt.dayofweek
)

serie_roll["mes"] = (
    serie_roll["timestamp"]
    .dt.month
)

serie_roll["es_fin_semana"] = (
    serie_roll["dia_semana"] >= 5
)

serie_roll[[
    "timestamp",
    "hora",
    "dia_semana",
    "mes",
    "es_fin_semana"
]].head()

# 28. Perfil horario

In [ ]:
perfil_hora = (
    serie_roll
    .groupby(
        "hora",
        as_index=False
    )
    .agg(
        demanda_media=("demanda", "mean")
    )
)

perfil_hora

## 28.1 Visualizar perfil horario

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    perfil_hora["hora"],
    perfil_hora["demanda_media"],
    marker="o"
)

ax.set(
    title="Perfil medio por hora del día",
    xlabel="Hora",
    ylabel="Demanda media"
)

plt.show()

### Interpretación

La feature `hora` recupera el patrón diario que ya observábamos visualmente en la serie.

# 29. Introducir anomalías

In [ ]:
serie_anom = serie_roll.copy()

indices_anomalia = [
    500,
    1200,
    2000,
    3000
]

serie_anom.loc[
    indices_anomalia,
    "demanda"
] += [80, -70, 100, 90]

serie_anom.loc[
    indices_anomalia,
    ["timestamp", "demanda"]
]

# 30. Visualizar anomalías antes de detectarlas

In [ ]:
tramo_anom = serie_anom.iloc[
    450:550
]

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(
    tramo_anom["timestamp"],
    tramo_anom["demanda"]
)

ax.scatter(
    serie_anom.loc[[500], "timestamp"],
    serie_anom.loc[[500], "demanda"]
)

ax.set(
    title="Anomalía introducida deliberadamente",
    xlabel="Fecha",
    ylabel="Demanda"
)

plt.show()

# 31. Detección rolling

In [ ]:
serie_anom["media_prev_24h"] = (
    serie_anom["demanda"]
    .shift(1)
    .rolling(
        24,
        min_periods=12
    )
    .mean()
)

serie_anom["std_prev_24h"] = (
    serie_anom["demanda"]
    .shift(1)
    .rolling(
        24,
        min_periods=12
    )
    .std()
)

serie_anom["z_prev_24h"] = (
    (
        serie_anom["demanda"]
        - serie_anom["media_prev_24h"]
    )
    /
    serie_anom["std_prev_24h"]
)

serie_anom["es_anomalia"] = (
    serie_anom["z_prev_24h"]
    .abs()
    > 3
)

serie_anom.loc[
    serie_anom["es_anomalia"],
    [
        "timestamp",
        "demanda",
        "media_prev_24h",
        "z_prev_24h"
    ]
].head(20)

### Punto clave

La media y desviación se calculan sobre:

```python
shift(1).rolling(...)
```

Es decir, **solo información anterior al instante actual**.

# 32. Visualizar detección

In [ ]:
muestra_det = serie_anom.iloc[
    450:550
]

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    muestra_det["timestamp"],
    muestra_det["demanda"],
    label="Demanda"
)

anomalias_muestra = muestra_det.loc[
    muestra_det["es_anomalia"]
]

ax.scatter(
    anomalias_muestra["timestamp"],
    anomalias_muestra["demanda"],
    label="Anomalía"
)

ax.legend()
ax.set(
    title="Detección de anomalías con ventana histórica",
    xlabel="Fecha",
    ylabel="Demanda"
)

plt.show()

# 33. Leakage temporal: provocar el error conceptual

In [ ]:
serie_leak = serie_anom.copy()

serie_leak["media_centrada_24h"] = (
    serie_leak["demanda"]
    .rolling(
        window=24,
        center=True,
        min_periods=1
    )
    .mean()
)

serie_leak[[
    "timestamp",
    "demanda",
    "media_centrada_24h"
]].head()

### ¿Dónde está el problema?

Una ventana centrada utiliza observaciones anteriores **y posteriores** al instante actual.

Si esta feature se utiliza para entrenar un modelo predictivo, introduce información futura.

Eso es **data leakage temporal**.

# 34. Comparar feature con leakage vs feature válida

In [ ]:
comparacion_leak = serie_leak.head(
    24 * 7
)

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    comparacion_leak["timestamp"],
    comparacion_leak["demanda"],
    alpha=0.35,
    label="Demanda"
)

ax.plot(
    comparacion_leak["timestamp"],
    comparacion_leak["media_centrada_24h"],
    label="Rolling centrado (leakage)"
)

ax.plot(
    comparacion_leak["timestamp"],
    serie_anom.head(24 * 7)["media_prev_24h"],
    label="Rolling histórico"
)

ax.legend()
ax.set(
    title="Feature con leakage vs feature histórica",
    xlabel="Fecha",
    ylabel="Demanda"
)

plt.show()

### Interpretación

La feature centrada suele parecer “mejor” porque utiliza información que en producción todavía no existiría.

Una feature más ajustada visualmente no es necesariamente más válida.

# 35. Validación explícita contra leakage

In [ ]:
def validar_feature_historica(
    datos: pd.DataFrame,
    feature: str
) -> bool:
    return (
        datos[feature]
        .notna()
        .sum()
        > 0
    )

print(
    "Feature histórica construida:",
    validar_feature_historica(
        serie_anom,
        "media_prev_24h"
    )
)

print(
    "La ventana centrada NO es válida para predicción causal."
)

# 36. Rolling por grupo

In [ ]:
serie_regiones = pd.DataFrame({
    "timestamp": pd.date_range(
        "2025-01-01",
        periods=24 * 60,
        freq="h"
    ).repeat(3),
    "region": np.tile(
        ["Norte", "Sur", "Este"],
        24 * 60
    )
})

serie_regiones["demanda"] = (
    100
    + rng.normal(
        0,
        10,
        len(serie_regiones)
    )
)

serie_regiones = (
    serie_regiones
    .sort_values(
        ["region", "timestamp"]
    )
    .reset_index(drop=True)
)

serie_regiones.head()

## 36.1 Rolling por región

In [ ]:
serie_regiones["media_24h"] = (
    serie_regiones
    .groupby("region")["demanda"]
    .transform(
        lambda s:
            s.shift(1)
            .rolling(
                24,
                min_periods=12
            )
            .mean()
    )
)

serie_regiones.head(30)

### Interpretación

Cada región debe tener su propia ventana histórica. Mezclar observaciones de regiones distintas destruiría la semántica temporal del grupo.

# 37. Preparar Polars

In [ ]:
if POLARS_DISPONIBLE:
    serie_pl = pl.from_pandas(
        serie_roll[[
            "timestamp",
            "demanda",
            "hora",
            "dia_semana"
        ]]
    )

    print(serie_pl.head())
else:
    print("Instalar Polars para continuar.")

# 38. Features temporales en Polars

In [ ]:
if POLARS_DISPONIBLE:
    features_pl = (
        serie_pl
        .with_columns(
            pl.col("timestamp")
            .dt.hour()
            .alias("hora_pl"),

            pl.col("timestamp")
            .dt.weekday()
            .alias("dia_semana_pl"),

            pl.col("timestamp")
            .dt.month()
            .alias("mes_pl")
        )
    )

    print(features_pl.head())

Polars utiliza expresiones temporales sobre `pl.col("timestamp").dt...`.

# 39. Lag en Polars

In [ ]:
if POLARS_DISPONIBLE:
    lag_pl = (
        serie_pl
        .with_columns(
            pl.col("demanda")
            .shift(1)
            .alias("lag_1h"),

            pl.col("demanda")
            .shift(24)
            .alias("lag_24h")
        )
    )

    print(lag_pl.head(30))

# 40. Diferencias en Polars

In [ ]:
if POLARS_DISPONIBLE:
    diff_pl = (
        serie_pl
        .with_columns(
            pl.col("demanda")
            .diff(1)
            .alias("diff_1h"),

            pl.col("demanda")
            .diff(24)
            .alias("diff_24h")
        )
    )

    print(diff_pl.head(30))

# 41. Rolling mean en Polars

In [ ]:
if POLARS_DISPONIBLE:
    rolling_pl = (
        serie_pl
        .with_columns(
            pl.col("demanda")
            .rolling_mean(
                window_size=24
            )
            .alias("media_24h")
        )
    )

    print(rolling_pl.head(30))

### Diferencia importante

`rolling_mean(window_size=24)` trabaja por número de filas.

Para ventanas temporales reales o agrupaciones dinámicas, Polars ofrece otras operaciones específicas.

# 42. Group by dynamic en Polars

In [ ]:
if POLARS_DISPONIBLE:
    diario_pl = (
        serie_pl
        .group_by_dynamic(
            "timestamp",
            every="1d"
        )
        .agg(
            pl.col("demanda")
            .mean()
            .alias("media"),

            pl.col("demanda")
            .min()
            .alias("minimo"),

            pl.col("demanda")
            .max()
            .alias("maximo"),

            pl.col("demanda")
            .sum()
            .alias("total")
        )
        .sort("timestamp")
    )

    print(diario_pl.head())

`group_by_dynamic()` permite construir ventanas temporales periódicas y cumple un papel equivalente a muchos usos de `resample()` en Pandas.

# 43. Validar resampling Pandas vs Polars

In [ ]:
if POLARS_DISPONIBLE:
    diario_pd = (
        serie_roll
        .set_index("timestamp")["demanda"]
        .resample("D")
        .agg(
            media="mean",
            minimo="min",
            maximo="max",
            total="sum"
        )
        .reset_index()
    )

    diario_pl_pd = (
        diario_pl
        .to_pandas()
    )

    comparacion_diaria = (
        diario_pd
        .merge(
            diario_pl_pd,
            on="timestamp",
            suffixes=("_pd", "_pl")
        )
    )

    comparacion_diaria.head()

## 43.1 Equivalencia numérica

In [ ]:
if POLARS_DISPONIBLE:
    for col in [
        "media",
        "minimo",
        "maximo",
        "total"
    ]:
        print(
            col,
            np.allclose(
                comparacion_diaria[f"{col}_pd"],
                comparacion_diaria[f"{col}_pl"]
            )
        )

# 44. Rolling histórico en Polars

In [ ]:
if POLARS_DISPONIBLE:
    anom_pl = pl.from_pandas(
        serie_anom[[
            "timestamp",
            "demanda"
        ]]
    )

    anom_pl = (
        anom_pl
        .with_columns(
            pl.col("demanda")
            .shift(1)
            .rolling_mean(
                window_size=24,
                min_samples=12
            )
            .alias("media_prev_24h"),

            pl.col("demanda")
            .shift(1)
            .rolling_std(
                window_size=24,
                min_samples=12
            )
            .alias("std_prev_24h")
        )
        .with_columns(
            (
                (
                    pl.col("demanda")
                    - pl.col("media_prev_24h")
                )
                / pl.col("std_prev_24h")
            )
            .alias("z_prev_24h")
        )
        .with_columns(
            (
                pl.col("z_prev_24h")
                .abs()
                > 3
            )
            .alias("es_anomalia")
        )
    )

    print(
        anom_pl
        .filter(
            pl.col("es_anomalia")
        )
        .head(20)
    )

# 45. Rolling por grupo en Polars

In [ ]:
if POLARS_DISPONIBLE:
    regiones_pl = pl.from_pandas(
        serie_regiones[
            ["timestamp", "region", "demanda"]
        ]
    )

    regiones_pl = (
        regiones_pl
        .with_columns(
            pl.col("demanda")
            .shift(1)
            .over("region")
            .rolling_mean(
                window_size=24,
                min_samples=12
            )
            .over("region")
            .alias("media_24h")
        )
    )

    print(regiones_pl.head(30))

### Interpretación

El patrón de ventanas por grupo se expresa mediante operaciones de ventana y `over("region")`.

# 46. Lazy temporal pipeline

In [ ]:
if POLARS_DISPONIBLE:
    temporal_lazy = (
        serie_pl
        .lazy()
        .with_columns(
            pl.col("timestamp")
            .dt.hour()
            .alias("hora"),

            pl.col("demanda")
            .shift(1)
            .alias("lag_1h")
        )
        .filter(
            pl.col("hora")
            .is_between(
                8,
                20
            )
        )
        .group_by_dynamic(
            "timestamp",
            every="1d"
        )
        .agg(
            pl.col("demanda")
            .mean()
            .alias("demanda_media"),

            pl.col("lag_1h")
            .mean()
            .alias("lag_medio")
        )
    )

    print(
        temporal_lazy.explain(
            optimized=True
        )
    )

## 46.1 Ejecutar

In [ ]:
if POLARS_DISPONIBLE:
    resultado_temporal_lazy = (
        temporal_lazy
        .collect()
    )

    print(
        resultado_temporal_lazy.head()
    )

# 47. Benchmark resampling Pandas vs Polars

In [ ]:
def medir(funcion, repeticiones=7):
    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(
            time.perf_counter()
            - inicio
        )

    return np.array(tiempos)


def pandas_resample():
    return (
        serie_roll
        .set_index("timestamp")["demanda"]
        .resample("D")
        .mean()
    )


def polars_dynamic():
    if not POLARS_DISPONIBLE:
        return None

    return (
        serie_pl
        .group_by_dynamic(
            "timestamp",
            every="1d"
        )
        .agg(
            pl.col("demanda")
            .mean()
            .alias("media")
        )
    )


t_pd = medir(
    pandas_resample
)

if POLARS_DISPONIBLE:
    t_pl = medir(
        polars_dynamic
    )
else:
    t_pl = np.array([np.nan])

benchmark_resample = pd.DataFrame({
    "motor": ["Pandas", "Polars"],
    "media_ms": [
        t_pd.mean() * 1000,
        t_pl.mean() * 1000
    ],
    "std_ms": [
        t_pd.std() * 1000,
        t_pl.std() * 1000
    ]
})

benchmark_resample

## 47.1 Visualizar benchmark

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.bar(
        benchmark_resample["motor"],
        benchmark_resample["media_ms"],
        yerr=benchmark_resample["std_ms"],
        capsize=4
    )

    ax.set(
        title="Agregación temporal diaria",
        ylabel="Tiempo medio (ms)"
    )

    plt.show()

# 48. Benchmark rolling Pandas vs Polars

In [ ]:
def pandas_rolling():
    return (
        serie_roll["demanda"]
        .rolling(24)
        .mean()
    )


def polars_rolling():
    if not POLARS_DISPONIBLE:
        return None

    return (
        serie_pl
        .select(
            pl.col("demanda")
            .rolling_mean(
                window_size=24
            )
        )
    )


t_roll_pd = medir(
    pandas_rolling
)

if POLARS_DISPONIBLE:
    t_roll_pl = medir(
        polars_rolling
    )
else:
    t_roll_pl = np.array([np.nan])

benchmark_rolling = pd.DataFrame({
    "motor": [
        "Pandas rolling",
        "Polars rolling"
    ],
    "media_ms": [
        t_roll_pd.mean() * 1000,
        t_roll_pl.mean() * 1000
    ]
})

benchmark_rolling

## 48.1 Visualizar

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.bar(
        benchmark_rolling["motor"],
        benchmark_rolling["media_ms"]
    )

    ax.set(
        title="Rolling mean 24 observaciones",
        ylabel="Tiempo medio (ms)"
    )

    ax.tick_params(
        axis="x",
        rotation=15
    )

    plt.show()

### Interpretación

Los benchmarks solo son comparables cuando:

- usan la misma granularidad;
- producen el mismo resultado;
- incluyen suficientes repeticiones;
- no mezclan coste de conversión con coste de operación sin declararlo.

# 49. Validar rolling Pandas vs Polars

In [ ]:
if POLARS_DISPONIBLE:
    rolling_pd = (
        serie_roll["demanda"]
        .rolling(24)
        .mean()
        .to_numpy()
    )

    rolling_pl_np = (
        serie_pl
        .select(
            pl.col("demanda")
            .rolling_mean(
                window_size=24
            )
            .alias("media")
        )["media"]
        .to_numpy()
    )

    mascara = (
        ~np.isnan(rolling_pd)
        & ~np.isnan(rolling_pl_np)
    )

    print(
        "Rolling equivalente:",
        np.allclose(
            rolling_pd[mascara],
            rolling_pl_np[mascara]
        )
    )

# 50. Mini-práctica integrada

Objetivo:

> Construir features temporales válidas para predicción de demanda diaria sin utilizar información futura.

Debe incluir:

- lag 1h;
- lag 24h;
- media histórica 24h;
- desviación histórica 24h;
- hora;
- día de semana;
- media diaria;
- detección de anomalías.

## 50.1 Pandas

In [ ]:
features_pd = (
    serie_roll[
        ["timestamp", "demanda"]
    ]
    .copy()
)

features_pd["lag_1h"] = (
    features_pd["demanda"]
    .shift(1)
)

features_pd["lag_24h"] = (
    features_pd["demanda"]
    .shift(24)
)

features_pd["media_prev_24h"] = (
    features_pd["demanda"]
    .shift(1)
    .rolling(
        24,
        min_periods=12
    )
    .mean()
)

features_pd["std_prev_24h"] = (
    features_pd["demanda"]
    .shift(1)
    .rolling(
        24,
        min_periods=12
    )
    .std()
)

features_pd["hora"] = (
    features_pd["timestamp"]
    .dt.hour
)

features_pd["dia_semana"] = (
    features_pd["timestamp"]
    .dt.dayofweek
)

features_pd.head(30)

## 50.2 Polars

In [ ]:
if POLARS_DISPONIBLE:
    features_pl = (
        serie_pl
        .with_columns(
            pl.col("demanda")
            .shift(1)
            .alias("lag_1h"),

            pl.col("demanda")
            .shift(24)
            .alias("lag_24h"),

            pl.col("demanda")
            .shift(1)
            .rolling_mean(
                window_size=24,
                min_samples=12
            )
            .alias("media_prev_24h"),

            pl.col("demanda")
            .shift(1)
            .rolling_std(
                window_size=24,
                min_samples=12
            )
            .alias("std_prev_24h"),

            pl.col("timestamp")
            .dt.hour()
            .alias("hora"),

            pl.col("timestamp")
            .dt.weekday()
            .alias("dia_semana")
        )
    )

    print(features_pl.head(30))

## 50.3 Validar equivalencia

In [ ]:
if POLARS_DISPONIBLE:
    features_pl_pd = (
        features_pl
        .to_pandas()
    )

    columnas_comparar = [
        "lag_1h",
        "lag_24h",
        "media_prev_24h",
        "std_prev_24h"
    ]

    for columna in columnas_comparar:
        a = features_pd[columna].to_numpy()
        b = features_pl_pd[columna].to_numpy()

        mascara = (
            ~np.isnan(a)
            & ~np.isnan(b)
        )

        print(
            columna,
            np.allclose(
                a[mascara],
                b[mascara]
            )
        )

## 50.4 Visualizar features

In [ ]:
muestra_feat = features_pd.iloc[
    24 * 5:
    24 * 12
]

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    muestra_feat["timestamp"],
    muestra_feat["demanda"],
    label="Demanda"
)

ax.plot(
    muestra_feat["timestamp"],
    muestra_feat["media_prev_24h"],
    label="Media histórica 24h"
)

ax.legend()
ax.set(
    title="Feature temporal construida sin leakage",
    xlabel="Fecha",
    ylabel="Demanda"
)

plt.show()

### Interpretación final

Una buena feature temporal no solo debe correlacionar con el objetivo.

También debe respetar la disponibilidad real de información en el instante de predicción.

# 51. Checklist temporal

- [ ] ¿La columna temporal tiene dtype correcto?
- [ ] ¿Hay fechas inválidas?
- [ ] ¿Hay duplicados temporales?
- [ ] ¿La serie está ordenada?
- [ ] ¿Conozco la frecuencia esperada?
- [ ] ¿He detectado gaps?
- [ ] ¿He documentado la imputación?
- [ ] ¿Resampling usa la agregación correcta?
- [ ] ¿La ventana se define por filas o por tiempo?
- [ ] ¿Uso `shift()` antes de rolling cuando necesito solo pasado?
- [ ] ¿He evitado ventanas centradas para predicción causal?
- [ ] ¿Los lags son realmente históricos?
- [ ] ¿Las features temporales son reproducibles?
- [ ] ¿He validado propiedades del resultado?
- [ ] ¿Pandas y Polars producen resultados equivalentes?
- [ ] ¿He medido rendimiento?
- [ ] ¿He separado coste de conversión y coste de ejecución?

# 52. Ejercicios finales

## Ejercicio 1 — Parsing
Crea fechas válidas e inválidas, conviértelas con `errors="coerce"` y diagnostica.

## Ejercicio 2 — Duplicados
Introduce duplicados temporales y decide cuál debería ser la clave real.

## Ejercicio 3 — Gaps
Elimina observaciones y detecta la duración de los huecos.

## Ejercicio 4 — Reindexado
Materializa huecos mediante una frecuencia completa.

## Ejercicio 5 — Imputación
Compara forward fill e interpolación temporal gráficamente.

## Ejercicio 6 — Resampling
Calcula media, suma, mínimo y máximo diario y semanal.

## Ejercicio 7 — Rolling
Compara rolling de 24 filas frente a rolling de 24 horas.

## Ejercicio 8 — Expanding
Construye media acumulada y explica su diferencia con rolling.

## Ejercicio 9 — Lags
Genera lag 1, 24 y 168 horas.

## Ejercicio 10 — Features calendáricas
Extrae hora, día, semana, mes y fin de semana.

## Ejercicio 11 — Anomalías
Introduce anomalías y detecta mediante z-score histórico.

## Ejercicio 12 — Leakage
Construye una feature centrada y explica por qué no es válida.

## Ejercicio 13 — Rolling por grupo
Calcula ventanas independientes para varias regiones.

## Ejercicio 14 — Polars temporal
Reproduce lags, diferencias y features calendáricas.

## Ejercicio 15 — `group_by_dynamic`
Reproduce un resampling diario y valida contra Pandas.

## Ejercicio 16 — Lazy
Construye un pipeline temporal lazy y examina su plan.

## Ejercicio 17 — Benchmark
Compara rolling y agregación temporal Pandas vs Polars.

## Ejercicio 18 — Reto final
Sobre al menos seis meses de datos horarios:

1. introduce gaps;
2. introduce duplicados;
3. introduce fechas inválidas;
4. diagnostica;
5. corrige;
6. genera resampling diario/semanal;
7. crea lags;
8. crea rolling histórico;
9. introduce anomalías;
10. detecta anomalías;
11. provoca leakage;
12. corrígelo;
13. implementa Pandas;
14. implementa Polars eager;
15. implementa Polars lazy;
16. valida equivalencia;
17. benchmarkea;
18. interpreta todas las visualizaciones.

# 53. Preguntas de reflexión

1. ¿Por qué el orden importa en series temporales?
2. ¿Qué diferencia hay entre ventana por filas y por tiempo?
3. ¿Qué decisión analítica introduce un resampling?
4. ¿Qué diferencia existe entre rolling y expanding?
5. ¿Qué es un lag?
6. ¿Por qué `shift(1).rolling(...)` puede ser importante?
7. ¿Qué es leakage temporal?
8. ¿Por qué una ventana centrada puede ser inválida?
9. ¿Cómo detectar gaps?
10. ¿Por qué la imputación debe documentarse?
11. ¿Qué hace `group_by_dynamic` en Polars?
12. ¿Qué ventajas ofrece un pipeline temporal lazy?
13. ¿Por qué debemos validar equivalencia entre motores?
14. ¿Un benchmark aislado basta para elegir motor?
15. ¿Qué condiciones debe cumplir una feature temporal para ser válida en producción?